In [1]:
# Import necessary libraries.
from sklearn.linear_model import LogisticRegression # Logistic Regression model
from sklearn.ensemble import RandomForestClassifier # Random Forest classifier
from sklearn.model_selection import train_test_split, GridSearchCV # Modules for splitting train/test data and for Grid Search
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif # Modules for feature selection (Select top K, Variance Threshold, ANOVA F-value)
from sklearn.tree import DecisionTreeClassifier # Decision Tree classifier
from sklearn.metrics import roc_auc_score, fbeta_score, make_scorer # ROC AUC score, F-beta score, create custom scorer
from xgboost import XGBClassifier # XGBoost classifier
import shap # SHAP(SHapley Additive exPlanations) library (explaining model predictions)
import matplotlib.pyplot as plt # Library for data visualization

import pandas as pd # Library for data manipulation and analysis
import numpy as np # Library for numerical calculations
import datetime as dt # Library for date and time handling
import json # Library for JSON data handling

In [2]:
# Module for handling warning messages
import warnings 

# Ignore only the 'use_label_encoder' warning.
warnings.filterwarnings("ignore")

#### prepare "data/initial_dataset.p"

In [ ]:
# .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3

# Specify the file path
file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.xlsx'

# Read the Excel file into a DataFrame
# By default, it reads the first sheet.
data_row = pd.read_excel(file_path)

# Define the file path
output_file_path = 'data/1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'

# Save the DataFrame to a pickle file
data_row.to_pickle(output_file_path)

In [4]:
# # .xlsx -> .p : 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress

# # Specify the file path
# file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.xlsx'

# # Read the Excel file into a DataFrame
# # By default, it reads the first sheet.
# data_row = pd.read_excel(file_path)

# # Define the file path
# output_file_path = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'

# # Save the DataFrame to a pickle file
# data_row.to_pickle(output_file_path)


In [5]:
# read *.p
pickle_file_path_1 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with 3Lots_FT1_FT2_FT3.p'
data_row_1 = pd.read_pickle(pickle_file_path_1)
pickle_file_path_2 = 'data\\1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP with QualWfrs_FT&QAwithQAAfterStress.p'
data_row_2 = pd.read_pickle(pickle_file_path_2)


In [6]:
# Keep only the necessary columns

# Specify the file path
file_path = 'data/cols_to_keep.csv'

# Read the CSV file into a DataFrame
cols_to_keep_df = pd.read_csv(file_path)

cols_to_keep = cols_to_keep_df.iloc[:, 0].tolist()

data_row_1 = data_row_1[cols_to_keep]


In [7]:
# data_row <= data_row1 data_row2

# Select only the columns to join from data_row_2
columns_to_join = ['DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP',
                   'BG pass/fail']

# Create a subset DataFrame of data_row_2 with the selected columns
data_row_2_subset = data_row_2[columns_to_join]

# Merge the selected columns from data_row_2 into data_row_1 using the 'DevID' join key
data_row = pd.merge(data_row_1, data_row_2_subset, on='DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP', how='left')


In [8]:
initial_dataset = data_row.copy() # Copy the original dataset

In [9]:
# prepare for target

initial_dataset['Pass/Fail_pass'] = ((initial_dataset['soft_bin of FT1'] == 1) &
                   (initial_dataset['soft_bin of FT2'] == 1) &
                   (initial_dataset['soft_bin'] == 1)).astype(int)


In [10]:
# prepare for base model

initial_dataset['band gap dpat'] = initial_dataset['BG pass/fail'].apply(lambda x: 'bandGapFail' if x == 'impossible wafer' else 'ok for band gap')

# Create a dictionary for column name changes
new_column_names = {
    'wafer_id': 'WAFER_NO',
    'DevID of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP_JPP': 'DevID'
}

# Rename the columns using the .rename() method (using inplace=True to apply directly to the original DataFrame)
initial_dataset.rename(columns=new_column_names, inplace=True)


In [11]:
initial_dataset


,WF of 1103959_69_1133529_cp1,ROW of 1103959_69_1133529_cp1,COL of 1103959_69_1133529_cp1,DevID,S of 1103959_69_1133529_cp1,F:E of 1103959_69_1133529_cp1,FAILING_BINOUTS(1) of 1103959_69_1133529_cp1,HH_CONT_OUT of 1103959_69_1133529_cp1,HH_CONT_CL of 1103959_69_1133529_cp1,HH_CONT_CM of 1103959_69_1133529_cp1,...,EEPROM_T_SCAL_G_BIAS[],EEPROM_V4[],EEPROM_BANK8_D[],EEPROM_BANK8_C[],EEPROM_BANK9_D[],EEPROM_BANK9_C[],EEPROM_LOCKPAT_check[],BG pass/fail,Pass/Fail_pass,band gap dpat
0,2,17,35,[110395 17 35],8,.:.,NaN,-0.3747,-0.3812,-0.3783,...,4.0,122.0,35.0,199.0,18.0,123.0,83.0,impossible wafer,1,bandGapFail
1,2,19,38,[110395 19 38],8,.:.,NaN,-0.3849,-0.3934,-0.3894,...,3.0,140.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap
2,2,-7,63,[110395 -7 63],8,.:.,NaN,-0.3859,-0.3815,-0.3792,...,243.0,128.0,35.0,199.0,18.0,123.0,83.0,impossible wafer,1,bandGapFail
3,2,13,70,[110395 13 70],8,.:.,NaN,-0.3928,-0.3833,-0.3811,...,4.0,96.0,36.0,71.0,18.0,123.0,83.0,NaN,1,ok for band gap
4,2,33,57,[110395 33 57],8,.:.,NaN,-0.3875,-0.3796,-0.3768,...,243.0,148.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4531,2,31,63,[113352 31 63],8,.:.,NaN,-0.3798,-0.4050,-0.4015,...,243.0,112.0,35.0,198.0,18.0,123.0,83.0,NaN,1,ok for band gap
4532,2,-7,29,[113352 -7 29],8,.:.,NaN,-0.3965,-0.4209,-0.4189,...,242.0,128.0,35.0,199.0,18.0,123.0,83.0,pass,1,ok for band gap
4533,2,20,32,[113352 20 32],8,.:.,NaN,-0.3673,-0.4128,-0.4070,...,243.0,128.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap
4534,2,35,60,[113352 35 60],8,.:.,NaN,-0.3786,-0.4056,-0.4240,...,243.0,140.0,35.0,199.0,18.0,123.0,83.0,NaN,1,ok for band gap


In [12]:
# Define a list of columns to drop
columns_to_drop = [
    'soft_bin of FT1',
    'soft_bin of FT2',
    'soft_bin',
    'BG pass/fail'
]

# Drop columns (use inplace=True to modify the original DataFrame)
# Or create a new DataFrame with processed_dataset = processed_dataset.drop(...)
initial_dataset.drop(columns=columns_to_drop, inplace=True)


In [13]:
# Rename the columns
initial_dataset.rename(columns={'x_pos': 'X', 'y_pos': 'Y'}, inplace=True)


In [14]:
# Calculate the Radius column
# The np.sqrt() function calculates the square root of each element.
initial_dataset['Radius'] = np.sqrt(initial_dataset['X']**2 + initial_dataset['Y']**2)
initial_dataset.to_pickle("data/initial_dataset.p")


#### scnarios


In [15]:
# import vars and function

from algos.algos import *
from config.config import *


In [16]:
import copy


##### preprocess_dataset


In [17]:
preprocessed_dataset = preprocess_dataset(initial_dataset)




      Preprocessing dataset...
      Preprocessing complete!



##### create_train_and_test_data


In [18]:
split_parameter = copy.deepcopy(split_parameter_default)
train_data, test_data, split_parameter_info = create_train_test_data(preprocessed_dataset, split_parameter)




##############################################################################################################################
# 3) Create Train/Test Split 
##############################################################################################################################

      Creating training and test datasets...
    - Not applying filtering before splitting.
    - Not applying Feature Generation.

    - Training data class distribution before splitting: {0.0: 3546, 1.0: 71}
    - No sampling applied


##### Use user-specified thresholds - feature selection and feature importance calculation


##### select_feature


In [19]:
feature_selector_params_var = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_var["filter_methods"]["apply_variance_filter"] = True
feature_selector_params_var["filter_methods"]["var_threshold"] = 0.00
feature_selection_info_var = select_feature(train_data, feature_selector_params_var)

feature_selector_params_licor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_licor["filter_methods"]["apply_target_linear_corr_filter"] = True
feature_selector_params_licor["filter_methods"]["target_linear_corr_threshold"] = 0.00
feature_selection_info_licor = select_feature(train_data, feature_selector_params_licor)

feature_selector_params_xicor = copy.deepcopy(feature_selector_params_FeatureFilter_default)
feature_selector_params_xicor["filter_methods"]["apply_target_xicor_filter"] = True
feature_selector_params_xicor["filter_methods"]["target_xicor_threshold"] = 0.00
feature_selection_info_xicor = select_feature(train_data, feature_selector_params_xicor)

feature_selector_params_sfm = copy.deepcopy(feature_selector_params_sfm_default)
feature_selector_params_sfm["params"]["estimator"]["params"]["n_estimators"] = 250 # max = len(train_data.columns) - 1
feature_selector_params_sfm["params"]["threshold"] = "0*median"
feature_selection_info_sfm = select_feature(train_data, feature_selector_params_sfm)

feature_selection_infos = {
     "var" : feature_selection_info_var,
     "licor" : feature_selection_info_licor,
     "xicor" : feature_selection_info_xicor,
     "model" : feature_selection_info_sfm
}
for fileter_name, feature_selection_info in feature_selection_infos.items():
    print("# of feature:", feature_selection_info["final_feature_count"], ",  filter: ", feature_selection_info["feature_selector_name"], fileter_name)




--- Feature Selector: FeatureFilter ---

--- Starting Feature Filtering ---
    - Number of features remaining after variance filtering: 1401

Feature selection results saved to 'data/result/jsons\feature_selection_info_250903_115616_598abcd5.json' file.

- Final feature count: 1401

--- Feature Selector: FeatureFilter ---

--- Starting Feature Filtering ---
    - Number of features remaining after target linear correlation filtering: 1650

Feature selection results saved to 'data/result/jsons\feature_selection_info_250903_115617_a7da81a1.json' file.

- Final feature count: 1650

--- Feature Selector: FeatureFilter ---

--- Starting Feature Filtering ---
    - 타겟 Xi Cor 필터링 후 남은 피처 수: 1650

Feature selection results saved to 'data/result/jsons\feature_selection_info_250903_115617_ac7040df.json' file.

- Final feature count: 1650

--- Feature Selector: SFM ---
--- SFM Selector completed ---
Number of remaining features: 1650

Feature selection results saved to 'data/result/jsons\featur

In [20]:
# Summarize feature selector importance information

features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    features_values_dfs = pd.merge(
        features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )


In [21]:
# # Feature selection results by method: dic

feature_selection_info


{'feature_selector_name': 'SFM',
 'params': {'threshold': '0*median',
  'estimator': {'name': 'RandomForestClassifier',
   'params': {'n_estimators': 250, 'max_depth': 12}}},
 'filter_methods': 'apply_SelectFromModel_filter',
 'initial_feature_count': 1650,
 'final_feature_count': 1650,
 'final_features': ['X',
  'Y',
  'Radius',
  'AC_COIL_FACTOR of 1103959_69_1133529_YPP',
  'AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5',
  'AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5_YPP_QPP',
  'AC_COIL_FACTOR of 1103959_69_1133592_RPP',
  'AC_GAIN_32 of 1103959_69_1133529_cp1',
  'AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP',
  'AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP',
  'AC_GAIN_32 of 1103959_69_1133529_cp1p5',
  'AC_GAIN_32 of 1103959_69_1133592_QPP',
  'AC_GAIN_32 of 1103959_69_JPP',
  'AC_GAIN_33 of 1103959_69_1133529_cp1',
  'AC_GAIN_33 of 1103959_69_1133529_cp1_cp1p5_YPP',
  'AC_GAIN_33 of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP',
  'AC_GAIN_33 of 1103959_69_1133529_

In [22]:
# # Feature selection results summary data by method: df

features_values_dfs


,feature_name,FeatureFilter_variance,FeatureFilter_target_linear_correlation,FeatureFilter_target_xicor_correlation,SFM_importances
0,AC_COIL_FACTOR of 1103959_69_1133529_YPP,2.951574e-33,0.021211,0.255277,0.000000
1,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,2.951574e-33,0.021211,0.255277,0.000000
2,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...,2.951574e-33,0.021211,0.255277,0.000000
3,AC_COIL_FACTOR of 1103959_69_1133592_RPP,2.951574e-33,0.021211,0.255277,0.000000
4,AC_GAIN_32 of 1103959_69_1133529_cp1,0.000000e+00,NaN,0.456388,0.000000
...,...,...,...,...,...
1646,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1_cp1...,1.000277e+00,0.017764,0.505893,0.000395
1647,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1p5,1.000277e+00,0.026030,0.396025,0.000935
1648,Zb_V_OBVOL_VSS_L of 1103959_69_1133592_QPP,1.000277e+00,0.001384,0.447422,0.000143
1649,Zb_V_OBVOL_VSS_L of 1103959_69_JPP,1.000277e+00,0.067453,0.604254,0.000389


In [27]:
# feature select test : pipeline
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    # trained_model, feature_importance, train_parameters_info = train_model_xgboost_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


##### Use user-specified thresholds - feature selection > performance check: Use a simple model pipeline


In [28]:
# Apply the model based on the specified threshold and summarize the results
# feature select test - submit and summary result

usr_pl_fs_test_result_ftpn_df = pd.DataFrame()
usr_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for fileter_name, feature_selection_info in feature_selection_infos.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_test(
            train_data,
            feature_selection_info,
            train_parameters_list_default["rf_cv"],
            # train_parameters_list_default["xgboost_cv"],
            test_data
            )

    dict_ftpn = metrics["dict_ftpn"]
    
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'filter_methods' : feature_selection_info["filter_methods"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'feature_selection_info_json_path' : feature_selection_info["feature_selection_info_json_path"]
    }
    
    usr_pl_fs_test_result_ftpn_df = pd.concat([usr_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)

    if feature_selection_info["feature_selector_name"] == 'FeatureFilter':
        if feature_selection_info["filter_methods"]["apply_variance_filter"] == True:
            value_type = "variance"
        elif feature_selection_info["filter_methods"]["apply_target_linear_corr_filter"] == True:
            value_type = "target_linear_correlation"
        elif feature_selection_info["filter_methods"]["apply_target_xicor_filter"] == True:
            value_type = "target_xicor_correlation"
        features_values = feature_selection_info["selection_details"][value_type]["features_values_checked"]
    else : 
        value_type = "importances"
        features_values = feature_selection_info["selection_details"][value_type]

    features_values_df = pd.DataFrame(
        list(features_values.items()), 
        columns=['feature_name', feature_selection_info["feature_selector_name"]+"_"+value_type]
    )

    usr_pl_fs_test_result_features_values_dfs = pd.merge(
        usr_pl_fs_test_result_features_values_dfs,
        features_values_df,
        how='outer',
        left_on='feature_name',
        right_on='feature_name'
        )


      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 50}
    Best F2 (class=1) score (CV): 0.2258

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.7172 with F2 score: 0.6153
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Creating the metrics...
      Training the Random Forest model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

    Best parameters found: {'max_depth': 5, 'min_samples_split': 2, 'n_estimators': 20}
    Best F2 (class=1) score (CV): 0.2240

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.6970 with F2 score: 0.5909
     Calculation of the ROC curve...
     Calculation done
     Scoring...
    

In [29]:
# Model performance summary of the feature selection set - based on user feature selection threshold

usr_pl_fs_test_result_ftpn_df


,fn,fp,tn,tp,feature_selector_name,filter_methods,initial_feature_count,final_feature_count,feature_selection_info_json_path
0,5,180,707,13,FeatureFilter,"{'apply_variance_filter': True, 'var_threshold...",1650,1401,data/result/jsons\feature_selection_info_25090...
1,6,184,703,12,FeatureFilter,"{'apply_variance_filter': False, 'var_threshol...",1650,1650,data/result/jsons\feature_selection_info_25090...
2,6,184,703,12,FeatureFilter,"{'apply_variance_filter': False, 'var_threshol...",1650,1650,data/result/jsons\feature_selection_info_25090...
3,6,184,703,12,SFM,apply_SelectFromModel_filter,1650,1650,data/result/jsons\feature_selection_info_25090...


In [30]:
# Model performance summary of the feature selection set + applied columns from user feature selection threshold (for reference)

usr_pl_fs_test_result_features_values_dfs


,feature_name,FeatureFilter_variance,FeatureFilter_target_linear_correlation,FeatureFilter_target_xicor_correlation,SFM_importances
0,AC_COIL_FACTOR of 1103959_69_1133529_YPP,2.951574e-33,0.021211,0.255277,0.000000
1,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,2.951574e-33,0.021211,0.255277,0.000000
2,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...,2.951574e-33,0.021211,0.255277,0.000000
3,AC_COIL_FACTOR of 1103959_69_1133592_RPP,2.951574e-33,0.021211,0.255277,0.000000
4,AC_GAIN_32 of 1103959_69_1133529_cp1,0.000000e+00,NaN,0.456388,0.000000
...,...,...,...,...,...
1646,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1_cp1...,1.000277e+00,0.017764,0.505893,0.000395
1647,Zb_V_OBVOL_VSS_L of 1103959_69_1133529_cp1p5,1.000277e+00,0.026030,0.396025,0.000935
1648,Zb_V_OBVOL_VSS_L of 1103959_69_1133592_QPP,1.000277e+00,0.001384,0.447422,0.000143
1649,Zb_V_OBVOL_VSS_L of 1103959_69_JPP,1.000277e+00,0.067453,0.604254,0.000389


##### Use feature importance information - optimal selection


###### The optimal selection method uses the XGBClassifier model and GridSearchCV for searching for optimal values


In [31]:
feature_importance_df = features_values_dfs.copy()


In [32]:
### Apply

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import fbeta_score, make_scorer, confusion_matrix
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
import uuid


In [33]:
# train_data


In [34]:
# Select the target as the rightmost column
target_df = train_data.iloc[:, -1]
# The rest of the columns are features
features_df = train_data.iloc[:, :-1]

X_train, X_test, y_train, y_test = train_test_split(features_df, target_df, test_size=0.2, random_state=42)


In [35]:
# run_optimization_for_feature_importance : a function to select features based on a specific importance column and find the optimal model

def run_optimization_for_feature_importance(train_data, target_data, feature_importance_df, importance_column, k_percentiles):
    """
    A function to select features based on a specific importance column and find the optimal model.
    
    Parameters:
    - train_data (pd.DataFrame): Training data
    - target_data (pd.Series): Target data
    - feature_importance_df (pd.DataFrame): DataFrame containing feature importance information
    - importance_column (str): The column name to determine the importance rank
    - k_percentiles (list): List of candidate percentiles for the number of features to select (e.g., [0.05, 0.1, 0.25, 0.5])

    Returns:
    - pd.DataFrame: Optimized feature importance information
    - pd.DataFrame: Model performance summary information
    """
    f2_scorer = make_scorer(fbeta_score, beta=2.0)
    
    # Select the top K features based on the values in the importance column
    sorted_features = feature_importance_df.sort_values(
        by=importance_column, ascending=False
    )['feature_name']
    
    # Convert percentiles to the actual number of features
    n_features_total = len(sorted_features)
    k_values = [max(1, int(n_features_total * p)) for p in k_percentiles]
    
    best_k = k_values[0]
    best_score = -1.0
    best_pipeline = None
    selected_feature_list = []
    # ⭐️ Variable to store the optimal percentile value
    best_k_percentile = k_percentiles[0]

    for i, k in enumerate(k_values):
        top_k_features = sorted_features.head(k).tolist()
        
        # Prepare the dataset with only the optimal features
        X_train_filtered = train_data[top_k_features]
        
        # Model training pipeline (direct feature selection instead of SelectKBest)
        pipeline = Pipeline([
            ('model', XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'))
        ])
        
        param_grid = {
            'model__n_estimators': [50, 100],
            'model__max_depth': [3, 5]
        }
        
        grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring=f2_scorer, n_jobs=-1)
        grid_search.fit(X_train_filtered, target_data)

        # Evaluate the performance for the current K
        if grid_search.best_score_ > best_score:
            best_score = grid_search.best_score_
            best_k = k
            best_pipeline = grid_search.best_estimator_
            selected_feature_list = top_k_features
            # ⭐️ Update the best percentile
            best_k_percentile = k_percentiles[i]

    # Generate feature importance and performance information for the optimal model
    best_xgb_model = best_pipeline.named_steps['model']
    
    feature_info = pd.DataFrame({
        'feature_name': train_data.columns
    })
    feature_info['is_selected'] = feature_info['feature_name'].isin(selected_feature_list)
    feature_info['importance_column'] = importance_column
    feature_importances = {name: 0 for name in train_data.columns}
    
    # Assign importance scores only to the selected features
    for i, importance in enumerate(best_xgb_model.feature_importances_):
        if i < len(selected_feature_list):
            feature_importances[selected_feature_list[i]] = importance
        
    feature_info['importance_score'] = feature_info['feature_name'].map(feature_importances)
    feature_info['feature_value_by_importance_column'] = feature_info['feature_name'].map(
        feature_importance_df.set_index('feature_name')[importance_column]
    )

    # Final performance evaluation with the test data
    y_pred = best_pipeline.predict(X_test[selected_feature_list])
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    performance_summary = pd.DataFrame([{
        'fn': fn,
        'fp': fp,
        'tn': tn,
        'tp': tp,
        'feature_selector_name': importance_column,
        'initial_feature_count': X_train.shape[1],
        'final_feature_count': len(selected_feature_list),
        'f2_score': fbeta_score(y_test, y_pred, beta=2.0),
        'importance_column': importance_column,
        # ⭐️ Add the optimal percentile column
        'best_k_percentile': best_k_percentile
    }])

    return feature_info, performance_summary


In [36]:
# Repeat optimization for each importance column and accumulate results
# Save the importance column names from the 2nd column onwards, excluding the feature name column
importance_cols = feature_importance_df.columns[1:].tolist()

# ⭐️ Change to a list of percentile candidates: defined by the user
k_percentiles = [0.05, 0.1, 0.25, 0.5, 0.75, 0.8, 0.85, 0.9]

all_feature_infos = []
all_performance_summaries = []

for col in importance_cols:
    print(f"\n--- Running optimization based on {col} column ---")
    feat_info, perf_summary = run_optimization_for_feature_importance(
        X_train, y_train, feature_importance_df, col, k_percentiles
    )
    all_feature_infos.append(feat_info)
    all_performance_summaries.append(perf_summary)

final_feature_info_df = pd.concat(all_feature_infos, ignore_index=True)
final_feature_performance_summary_df = pd.concat(all_performance_summaries, ignore_index=True)

# 4. Print results and save to CSV
print("\n--- Final accumulated feature importance information ---")
print(final_feature_info_df.head(10))
final_feature_info_df.to_csv('final_feature_info.csv', index=False)

print("\n--- Final accumulated model performance summary ---")
print(final_feature_performance_summary_df)
final_feature_performance_summary_df.to_csv('final_feature_performance_summary.csv', index=False)



--- Running optimization based on FeatureFilter_variance column ---

--- Running optimization based on FeatureFilter_target_linear_correlation column ---

--- Running optimization based on FeatureFilter_target_xicor_correlation column ---

--- Running optimization based on SFM_importances column ---

--- Final accumulated feature importance information ---
                                        feature_name  is_selected  \
0                                                  X         True   
1                                                  Y         True   
2                                             Radius         True   
3           AC_COIL_FACTOR of 1103959_69_1133529_YPP        False   
4     AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5        False   
5  AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5...        False   
6           AC_COIL_FACTOR of 1103959_69_1133592_RPP        False   
7               AC_GAIN_32 of 1103959_69_1133529_cp1        False   
8     AC_GAIN_32 of

In [37]:
# Automatic (optimal) feature selection result data: df
final_feature_info_df


,feature_name,is_selected,importance_column,importance_score,feature_value_by_importance_column
0,X,True,FeatureFilter_variance,0.000000,1.000277e+00
1,Y,True,FeatureFilter_variance,0.007534,1.000277e+00
2,Radius,True,FeatureFilter_variance,0.003558,1.000277e+00
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,False,FeatureFilter_variance,0.000000,2.951574e-33
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,False,FeatureFilter_variance,0.000000,2.951574e-33
...,...,...,...,...,...
6595,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbSbPoPo,False,SFM_importances,0.000000,7.015850e-05
6596,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,False,SFM_importances,0.000000,0.000000e+00
6597,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,False,SFM_importances,0.000000,6.139330e-07
6598,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,False,SFM_importances,0.000000,0.000000e+00


In [38]:
# Automatic (optimal) feature selection result data summary: df
final_feature_performance_summary_df

,fn,fp,tn,tp,feature_selector_name,initial_feature_count,final_feature_count,f2_score,importance_column,best_k_percentile
0,14,7,703,0,FeatureFilter_variance,1650,1320,0.0,FeatureFilter_variance,0.8
1,14,7,703,0,FeatureFilter_target_linear_correlation,1650,165,0.0,FeatureFilter_target_linear_correlation,0.1
2,14,7,703,0,FeatureFilter_target_xicor_correlation,1650,1485,0.0,FeatureFilter_target_xicor_correlation,0.9
3,14,6,704,0,SFM_importances,1650,825,0.0,SFM_importances,0.5


##### Use feature importance information - optimal selection > performance check: Use a simple model pipeline

In [39]:
# Automatic (optimal) feature selection result data: df
final_feature_info_df


,feature_name,is_selected,importance_column,importance_score,feature_value_by_importance_column
0,X,True,FeatureFilter_variance,0.000000,1.000277e+00
1,Y,True,FeatureFilter_variance,0.007534,1.000277e+00
2,Radius,True,FeatureFilter_variance,0.003558,1.000277e+00
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,False,FeatureFilter_variance,0.000000,2.951574e-33
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,False,FeatureFilter_variance,0.000000,2.951574e-33
...,...,...,...,...,...
6595,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbSbPoPo,False,SFM_importances,0.000000,7.015850e-05
6596,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,False,SFM_importances,0.000000,0.000000e+00
6597,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,False,SFM_importances,0.000000,6.139330e-07
6598,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,False,SFM_importances,0.000000,0.000000e+00


In [40]:
# Prepare performance check pipeline input: feature_selection_results
feature_selection_results = {}

# Iterate line by line
for idx, row in final_feature_performance_summary_df.iterrows():
    feature_selection_result = {}
    feature_selection_result["feature_selector_idx"] = idx
    feature_selection_result["feature_selector_name"] = row['feature_selector_name']
    feature_selection_result["initial_feature_count"] = row['initial_feature_count']
    feature_selection_result["final_feature_count"] = row['final_feature_count']
    feature_selection_result.setdefault("Params", {})["f2_score"] = row['f2_score']
    feature_selection_result.setdefault("Params", {})["best_k_percentile"] = row['best_k_percentile']
    
    feature_name_list = final_feature_info_df[
        (final_feature_info_df["importance_column"] == row['feature_selector_name']) &
        (final_feature_info_df["is_selected"] == True)
    ]["feature_name"].tolist()
    feature_selection_result["final_features"] = feature_name_list

    feature_selection_results[idx] = feature_selection_result

In [41]:
# final_feature_performance_summary_df

In [42]:
# feature_selection_results

In [43]:
# feature select test : pipeline
## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

def pl_fs_final_test(
        train_data,
        feature_selection_info,
        train_parameters,
        test_data
        ):
    
    # trained_model, feature_importance, train_parameters_info = train_model_rf_cv(train_data.copy(), feature_selection_info, train_parameters)
    trained_model, feature_importance, train_parameters_info = train_model_xgboost_cv(train_data.copy(), feature_selection_info, train_parameters)
    forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
    train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
    roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
    train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
    metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
    results = create_results(forecast_dataset, test_data, best_threshold)
    
    return \
        trained_model, \
        feature_importance, \
        forecast_dataset, \
        train_dataset_proba, \
        best_threshold, \
        roc_data, \
        auc_score, \
        train_dataset_metrics, \
        metrics, \
        results


In [44]:
# Optimal feature set performance check
# feature select test - submit and summary result

trained_model_set = {}
feature_importance_set = pd.DataFrame()
forecast_dataset_set = pd.DataFrame()
train_dataset_proba_set = pd.DataFrame()
best_threshold_set = pd.DataFrame()
roc_data_set = pd.DataFrame()
auc_score_set = pd.DataFrame()
train_dataset_metrics_set = pd.DataFrame()
metrics_set = pd.DataFrame()
results_set = pd.DataFrame()

opt_pl_fs_test_result_ftpn_df = pd.DataFrame()
opt_pl_fs_test_result_features_values_dfs = pd.DataFrame(
    list(train_data.columns), 
    columns=['feature_name']
)

for feature_selector_idx, feature_selection_info in feature_selection_results.items():
    trained_model, \
    feature_importance, \
    forecast_dataset, \
    train_dataset_proba, \
    best_threshold, \
    roc_data, \
    auc_score, \
    train_dataset_metrics, \
    metrics, \
    results \
    = \
    pl_fs_final_test(
            train_data,
            feature_selection_info,
            # train_parameters_list_default["rf_cv"],
            train_parameters_list_default["xgboost_cv"],
            test_data
            )

    trained_model_set[feature_selector_idx] = trained_model

    feature_importance["feature_selector_idx"] = feature_selector_idx
    feature_importance_set = pd.concat([feature_importance_set, feature_importance], ignore_index=True)

    forecast_dataset_df = pd.DataFrame({
        "feature_selector_idx": [feature_selector_idx] * len(forecast_dataset),
        "forecast_prob": forecast_dataset
    })
    forecast_dataset_set = pd.concat([forecast_dataset_set, forecast_dataset_df], ignore_index=True)
    
    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    train_dataset_proba["feature_selector_idx"] = feature_selector_idx
    train_dataset_proba_set = pd.concat([train_dataset_proba_set, train_dataset_proba], ignore_index=True)

    best_threshold_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "best_threshold": [best_threshold]
    })
    best_threshold_set = pd.concat([best_threshold_set, best_threshold_df], ignore_index=True)

    roc_data["feature_selector_idx"] = feature_selector_idx
    roc_data_set = pd.concat([roc_data_set, roc_data], ignore_index=True)

    auc_score_df = pd.DataFrame({
    "feature_selector_idx": [feature_selector_idx],
    "auc_score": [auc_score]
    })
    auc_score_set = pd.concat([auc_score_set, auc_score_df], ignore_index=True)

    train_dataset_metrics["feature_selector_idx"] = feature_selector_idx
    train_dataset_metrics_set = pd.concat([train_dataset_metrics_set, train_dataset_metrics], ignore_index=True)

    metrics_dict = {
        "f1_score": metrics["f1_score"],
        "recall": metrics["recall"],
        "precision": metrics["precision"],
        "accuracy": metrics["accuracy"],
        "auc_score": metrics["auc_score"],
        "tp": metrics["dict_ftpn"]["tp"],
        "tn": metrics["dict_ftpn"]["tn"],
        "fp": metrics["dict_ftpn"]["fp"],
        "fn": metrics["dict_ftpn"]["fn"],
        "number_of_predictions": metrics["number_of_predictions"],
        "number_of_good_predictions": metrics["number_of_good_predictions"],
        "number_of_false_predictions": metrics["number_of_false_predictions"],
        "feature_selector_idx": feature_selector_idx  # 현재 필터 이름 추가
    }
    metrics_df = pd.DataFrame([metrics_dict])
    metrics_set = pd.concat([metrics_set, metrics_df], ignore_index=True)

    results["feature_selector_idx"] = feature_selector_idx
    results_set = pd.concat([results_set, results], ignore_index=True)

    dict_ftpn = metrics["dict_ftpn"]
    new_row = {
        'fn': dict_ftpn.get('fn'),
        'fp': dict_ftpn.get('fp'),
        'tn': dict_ftpn.get('tn'),
        'tp': dict_ftpn.get('tp'),
        'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
        'feature_selector_name' : feature_selection_info["feature_selector_name"],
        'initial_feature_count' : feature_selection_info["initial_feature_count"],
        'final_feature_count' : feature_selection_info["final_feature_count"],
        'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
        'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
    }
    
    opt_pl_fs_test_result_ftpn_df = pd.concat([opt_pl_fs_test_result_ftpn_df, pd.DataFrame([new_row])], ignore_index=True)
    features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })
    opt_pl_fs_test_result_features_values_dfs = pd.merge(
        opt_pl_fs_test_result_features_values_dfs,
        features_values,
        how='left',
        left_on='feature_name',
        right_on='Features'
    ).drop('Features', axis=1) # The .drop() method is used to drop the column

import pickle
import os

save_path = "data/result/select_feature"
os.makedirs(save_path, exist_ok=True)

dict_vars = [
    (trained_model_set, "trained_model_set")
]

df_vars = [
    (feature_importance_set, "feature_importance_set"),
    (forecast_dataset_set, "forecast_dataset_set"),
    (train_dataset_proba_set, "train_dataset_proba_set"),
    (best_threshold_set, "best_threshold_set"),
    (roc_data_set, "roc_data_set"),
    (auc_score_set, "auc_score_set"),
    (train_dataset_metrics_set, "train_dataset_metrics_set"),
    (metrics_set, "metrics_set"),
    (results_set, "results_set"),
    (opt_pl_fs_test_result_ftpn_df, "opt_pl_fs_test_result_ftpn_df"),
    (opt_pl_fs_test_result_features_values_dfs, "opt_pl_fs_test_result_features_values_dfs")
]

for var, name in dict_vars:
    file_path = os.path.join(save_path, f"{name}.pickle")
    with open(file_path, "wb") as f:
        pickle.dump(var, f)
    print(f"Saved {name} to {file_path}")

for var, name in df_vars:
    file_path = os.path.join(save_path, f"{name}.csv")
    var.to_csv(file_path, index=False)
    print(f"Saved {name} to {file_path}")

     Training the XGBoost model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

     Best parameters found: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 100}
     Best F2 (class=1) score: 0.2360

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.8182 with F2 score: 0.8895
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Creating the metrics...
     Training the XGBoost model with cross-validation & hyperparameter tuning...

Fitting 3 folds for each of 18 candidates, totalling 54 fits

     Best parameters found: {'learning_rate': 0.2, 'max_depth': 2, 'n_estimators': 50}
     Best F2 (class=1) score: 0.2598

     Forecasting the test dataset...
     Forecasting done!
Best threshold for F2 score: 0.8384 with F2 score: 0.9285
     Calculation of the ROC curve...
     Calculation done
     Scoring...
     Scoring done

     Cr

In [45]:
opt_pl_fs_test_result_ftpn_df


,fn,fp,tn,tp,feature_selector_idx,feature_selector_name,initial_feature_count,final_feature_count,final_feature_selector_f2score,final_feature_selector_best_k_percentile
0,15,34,853,3,0,FeatureFilter_variance,1650,1320,0.0,0.8
1,16,21,866,2,1,FeatureFilter_target_linear_correlation,1650,165,0.0,0.1
2,16,28,859,2,2,FeatureFilter_target_xicor_correlation,1650,1485,0.0,0.9
3,15,33,854,3,3,SFM_importances,1650,825,0.0,0.5


In [46]:
opt_pl_fs_test_result_features_values_dfs

,feature_name,Importance(model)_FeatureFilter_variance,Importance(model)_FeatureFilter_target_linear_correlation,Importance(model)_FeatureFilter_target_xicor_correlation,Importance(model)_SFM_importances
0,X,0.000000,NaN,0.000000,0.012244
1,Y,0.046333,NaN,0.049318,NaN
2,Radius,0.010879,NaN,0.009446,0.018117
3,AC_COIL_FACTOR of 1103959_69_1133529_YPP,NaN,NaN,NaN,NaN
4,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,NaN,NaN,NaN,NaN
...,...,...,...,...,...
1646,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,NaN,NaN,0.000000,NaN
1647,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,NaN,NaN,0.000000,NaN
1648,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,NaN,NaN,0.000000,NaN
1649,band gap dpat_ok for band gap,NaN,NaN,0.000000,NaN


In [47]:
train_dataset_proba_set

,X,Y,Radius,AC_COIL_FACTOR of 1103959_69_1133529_YPP,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5_YPP_QPP,AC_COIL_FACTOR of 1103959_69_1133592_RPP,AC_GAIN_32 of 1103959_69_1133529_cp1,AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP,AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP,...,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,band gap dpat_ok for band gap,Pass/Fail,Probability,Historical,Forecast,True/False/Positive/Negative,feature_selector_idx
0,7.0,54.0,54.451814,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,False,False,False,True,0.0,0.239843,0.0,0,True Pass (TN),0
1,-16.0,39.0,42.154478,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,False,False,False,True,0.0,0.239843,0.0,0,True Pass (TN),0
2,-9.0,38.0,39.051248,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,True,False,False,0.0,0.737211,0.0,0,True Pass (TN),0
3,20.0,58.0,61.351447,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,False,False,False,True,0.0,0.239843,0.0,0,True Pass (TN),0
4,20.0,37.0,42.059482,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,False,False,False,True,0.0,0.239843,0.0,0,True Pass (TN),0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28931,19.0,68.0,70.604532,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,True,False,False,0.0,0.811645,0.0,0,True Pass (TN),3
28932,20.0,18.0,26.907248,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,False,False,False,True,0.0,0.239742,0.0,0,True Pass (TN),3
28933,28.0,19.0,33.837849,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,True,False,False,0.0,0.245205,0.0,0,True Pass (TN),3
28934,-15.0,39.0,41.785165,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,0.0,0.812672,0.0,0,True Pass (TN),3


In [48]:
test_data

,X,Y,Radius,AC_COIL_FACTOR of 1103959_69_1133529_YPP,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5,AC_COIL_FACTOR of 1103959_69_1133529_cp1_cp1p5_YPP_QPP,AC_COIL_FACTOR of 1103959_69_1133592_RPP,AC_GAIN_32 of 1103959_69_1133529_cp1,AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP,AC_GAIN_32 of 1103959_69_1133529_cp1_cp1p5_YPP_QPP_RPP,...,FAILING_BINOUTS(1) of 1103959_69_1133592_QPP_SbPo,FAILING_BINOUTS(1) of 1103959_69_1133592_QPP_SbPoPo,FAILING_BINOUTS(1) of 1103959_69_1133592_QPP_SbSbPoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbPoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_GbGbSbSbPoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_GbPoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_PoPo,FAILING_BINOUTS(1) of 1103959_69_JPP_SbPoPo,band gap dpat_ok for band gap,Pass/Fail
4373,16.0,14.0,21.260292,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,True,False,False,False,False,False,False,False,True,0.0
677,23.0,22.0,31.827661,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,False,False,True,False,True,0.0
57,19.0,51.0,54.424259,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,False,True,False,False,True,0.0
1344,43.0,44.0,61.522354,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,False,False,True,False,False,0.0
3835,-10.0,53.0,53.935146,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,True,False,False,False,False,False,False,False,True,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2898,41.0,39.0,56.586217,0.9738,0.9738,0.9738,0.9738,43.0,37.0,37.0,...,True,False,False,False,False,False,False,False,True,0.0
2366,37.0,22.0,43.046487,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,False,False,True,False,True,0.0
706,4.0,22.0,22.360680,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,False,False,True,False,False,0.0
805,15.0,52.0,54.120237,0.9738,0.9738,0.9738,0.9738,43.0,43.0,43.0,...,False,False,False,False,False,False,True,False,True,0.0


# Create Chart

##### chart for the group (feature_selector_idx = 0) which got lowest fn(fn:false nagative. actual is fail, prediction is pass) 

In [49]:
import pandas as pd

try:
    roc_data_set_df = pd.read_csv('data/result/select_feature/roc_data_set.csv')
    print("DataFrame 'roc_data_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(roc_data_set_df.head())
except FileNotFoundError:
    print("Error: The file 'data/result/select_feature/roc_data_set.csv' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'roc_data_set_df' has been successfully loaded.


In [50]:
roc_data_set_df

,False positive rate,True positive rate,feature_selector_idx
0,1.0,1.0,0
1,1.0,1.0,0
2,1.0,1.0,0
3,1.0,1.0,0
4,1.0,1.0,0
...,...,...,...
399,0.0,0.0,3
400,0.0,0.0,3
401,0.0,0.0,3
402,0.0,0.0,3


In [51]:
# !pip install -U kaleido

In [52]:
import pandas as pd
import plotly.express as px
from sklearn.metrics import auc
import numpy as np
import os

# roc_data_set_df가 이미 로드되었다고 가정합니다.

# 1. 'feature_selector_idx'가 0인 데이터 필터링
filtered_df = roc_data_set_df[roc_data_set_df['feature_selector_idx'] == 0].copy()

# 2. 곡선 완성을 위해 (0,0)과 (1,1) 포인트 추가
filtered_df.loc[-1] = {'False positive rate': 0.0, 'True positive rate': 0.0, 'feature_selector_idx': 0}
filtered_df.loc[len(filtered_df)] = {'False positive rate': 1.0, 'True positive rate': 1.0, 'feature_selector_idx': 0}
filtered_df.sort_values(by='False positive rate', inplace=True)

# 3. AUC 값 계산
roc_auc = auc(filtered_df['False positive rate'], filtered_df['True positive rate'])
roc_auc = round(roc_auc, 4)

# 4. Plotly Express로 차트 생성
fig = px.area(
    filtered_df,
    x='False positive rate',
    y='True positive rate',
    labels={'False positive rate': 'False Positive Rate', 'True positive rate': 'True Positive Rate'}
)

# 랜덤 분류기(점선) 추가
fig.add_scatter(
    x=[0, 1],
    y=[0, 1],
    mode='lines',
    line=dict(dash='dash', color='black'),
    name='Random Classifier'
)

# 5. 레이아웃 및 AUC 주석 설정
fig.update_layout(
    xaxis_title='False Positive Rate',
    yaxis_title='True Positive Rate',
    xaxis=dict(range=[0, 1], constrain='domain'),
    yaxis=dict(range=[0, 1], scaleanchor='x', scaleratio=1),
    showlegend=False,
    width=500,
    height=500,
    margin=dict(l=50, r=50, t=50, b=50),
    title_x=0.5,
    annotations=[
        dict(
            xref='paper', 
            yref='paper',
            x=0,          
            y=1.1,        
            text=f'ROC Curve (AUC={roc_auc})',
            showarrow=False,
            font=dict(size=14, color='black'),
            align='left'
        )
    ]
)

# 6. 지정된 폴더에 이미지 저장
save_path = "data/result/select_feature/ROC_Curve.png"

# 폴더가 없는 경우 생성
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# 이미지 파일로 저장
fig.write_image(save_path)

# VS Code에 차트 표시
fig.show()

print(f"ROC 차트가 {save_path}에 저장되었습니다.")

ROC 차트가 data/result/select_feature/ROC_Curve.png에 저장되었습니다.


In [53]:
import pandas as pd

try:
    results_set_df = pd.read_csv('data/result/select_feature/results_set.csv')
    print("DataFrame 'results_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(results_set_df.head())
except FileNotFoundError:
    print("Error: The file 'data/result/select_feature/results_set.csv' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'results_set_df' has been successfully loaded.


In [54]:
results_set_df

,Id,Probability,Forecast,Historical,True/False/Positive/Negative,feature_selector_idx
0,0,0.24,0,0.0,True Pass (TN),0
1,1,0.23,0,0.0,True Pass (TN),0
2,2,0.80,0,0.0,True Pass (TN),0
3,3,0.28,0,0.0,True Pass (TN),0
4,4,0.24,0,0.0,True Pass (TN),0
...,...,...,...,...,...,...
3615,900,0.24,0,0.0,True Pass (TN),3
3616,901,0.22,0,0.0,True Pass (TN),3
3617,902,0.40,0,0.0,True Pass (TN),3
3618,903,0.53,0,0.0,True Pass (TN),3


In [55]:
# import pandas as pd
# import plotly.express as px
# from sklearn.metrics import confusion_matrix
# import os

# # Assuming 'results_set_df' is loaded.
# # If not, load it:
# # results_set_df = pd.read_csv('data/result/select_feature/results_set.csv')

# # 1. Filter the DataFrame for 'feature_selector_idx' = 0
# filtered_df = results_set_df[results_set_df['feature_selector_idx'] == 0].copy()

# # 2. Correctly map string labels to binary classes (1 for Fail, 0 for Pass)
# # This is the core fix. The 'Historical' and 'Forecast' columns have specific text.
# y_true = filtered_df['True/False/Positive/Negative'].apply(lambda x: 1 if 'Fail' in x else 0)
# y_pred = filtered_df['True/False/Positive/Negative'].apply(lambda x: 1 if 'Fail' in x else 0)

# # Note: The provided image `image_7b1be2.png` only shows `True/False/Positive/Negative`
# # This single column seems to represent the result of a classification.
# # Based on the image, we can infer the following:
# # 'True Fail (TP)' -> Actual Fail (1), Predicted Fail (1)
# # 'Missed Fail (FN)' -> Actual Fail (1), Predicted Pass (0)
# # 'True Pass (TN)' -> Actual Pass (0), Predicted Pass (0)
# # There are no 'False Fail (FP)' examples in the sample, which means Actual Pass (0), Predicted Fail (1)

# # A more robust way to handle this is to use both 'Historical' and 'Forecast' as separate data.
# # However, since the provided image only shows one classification result column, we'll
# # deduce the true and predicted values from it.

# # Let's use the columns as they should be, based on standard ML practice
# y_actual = filtered_df['Historical']
# y_predicted = filtered_df['Forecast']

# # Re-mapping based on actual and predicted outcomes
# # Target is '1' (Failure).
# # Actuals: 1 for Fail, 0 for Pass
# # Predicted: 1 for Fail, 0 for Pass
# y_true_binary = y_actual.apply(lambda x: 1 if x == 1 else 0)
# y_pred_binary = y_predicted.apply(lambda x: 1 if x == 1 else 0)

# # 3. Calculate the confusion matrix
# # The `labels` argument ensures the order of the matrix is [Positive, Negative]
# cm = confusion_matrix(y_true_binary, y_pred_binary, labels=[1, 0])

# # 4. Create a DataFrame for the heatmap with correct labels
# cm_df = pd.DataFrame(
#     cm,
#     index=['True (Actual Fail)', 'False (Actual Pass)'],
#     columns=['Predicted Fail', 'Predicted Pass']
# )

# # 5. Plot the confusion matrix using Plotly Express
# fig = px.imshow(cm_df, text_auto=True, color_continuous_scale='blues')

# # Update layout for a better visualization
# fig.update_layout(
#     title='Confusion Matrix',
#     xaxis_title='Predicted Label',
#     yaxis_title='Actual Label',
#     width=500,
#     height=500,
#     xaxis_showgrid=False,
#     yaxis_showgrid=False
# )

# # Save the figure to a file
# save_path = "data/result/select_feature/confusion_matrix.png"
# os.makedirs(os.path.dirname(save_path), exist_ok=True)
# fig.write_image(save_path)

# fig.show()

# print(f"Corrected confusion matrix chart has been generated and saved to {save_path}.")

In [56]:
import pandas as pd
import plotly.express as px
from sklearn.metrics import confusion_matrix
import os

# Assuming 'results_set_df' is loaded.
# If not, load it:
# results_set_df = pd.read_csv('data/result/select_feature/results_set.csv')

# 1. Filter the DataFrame for 'feature_selector_idx' = 0
filtered_df = results_set_df[results_set_df['feature_selector_idx'] == 0].copy()

# A more robust way to handle this is to use both 'Historical' and 'Forecast' as separate data.
# However, since the provided image only shows one classification result column, we'll
# deduce the true and predicted values from it.

# Let's use the columns as they should be, based on standard ML practice
y_actual = filtered_df['Historical']
y_predicted = filtered_df['Forecast']

# Re-mapping based on actual and predicted outcomes
# Target is '1' (Failure).
# Actuals: 1 for Fail, 0 for Pass
# Predicted: 1 for Fail, 0 for Pass
y_true_binary = y_actual.apply(lambda x: 1 if x == 1 else 0)
y_pred_binary = y_predicted.apply(lambda x: 1 if x == 1 else 0)

# 3. Calculate the confusion matrix
# The `labels` argument ensures the order of the matrix is [Positive, Negative]
# The provided image shows Predicted Pass (0) before Predicted Fail (1) and
# True Pass (0) before True Fail (1). Let's adjust the labels accordingly.
cm = confusion_matrix(y_true_binary, y_pred_binary, labels=[0, 1])

# 4. Create a DataFrame for the heatmap with correct labels
# The provided image shows 'True label 0' (Pass) at the top and 'True label 1' (Fail) at the bottom.
# It also shows 'Predicted label 0' (Pass) on the left and 'Predicted label 1' (Fail) on the right.
# We need to reflect this order in our labels.
cm_df = pd.DataFrame(
    cm,
    index=['True (Actual Pass)', 'True (Actual Fail)'],
    columns=['Predicted Pass', 'Predicted Fail']
)

# 5. Plot the confusion matrix using Plotly Express
fig = px.imshow(cm_df, text_auto=True, color_continuous_scale='blues')

# Update layout for a better visualization
fig.update_layout(
    title='Train Set', # Title is now 'Train Set'
    xaxis_title='Predicted label',
    yaxis_title='True label',
    width=500,
    height=500,
    xaxis_showgrid=False,
    yaxis_showgrid=False
)

# Save the figure to a file
save_path = "data/result/select_feature/confusion_matrix.png"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
fig.write_image(save_path)

fig.show()

print(f"Corrected confusion matrix chart has been generated and saved to {save_path}.")

Corrected confusion matrix chart has been generated and saved to data/result/select_feature/confusion_matrix.png.


In [57]:
filtered_df

,Id,Probability,Forecast,Historical,True/False/Positive/Negative,feature_selector_idx
0,0,0.24,0,0.0,True Pass (TN),0
1,1,0.23,0,0.0,True Pass (TN),0
2,2,0.80,0,0.0,True Pass (TN),0
3,3,0.28,0,0.0,True Pass (TN),0
4,4,0.24,0,0.0,True Pass (TN),0
...,...,...,...,...,...,...
900,900,0.24,0,0.0,True Pass (TN),0
901,901,0.23,0,0.0,True Pass (TN),0
902,902,0.49,0,0.0,True Pass (TN),0
903,903,0.53,0,0.0,True Pass (TN),0


In [58]:
import pandas as pd

try:
    feature_importance_set_df = pd.read_csv('data/result/select_feature/feature_importance_set.csv')
    print("DataFrame 'feature_importance_set_df' has been successfully loaded.")
    # You can uncomment the line below to view the first 5 rows of the DataFrame
    # print(feature_importance_set_df.head())
except FileNotFoundError:
    print("Error: The file 'data/result/select_feature/feature_importance_set.csv' was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

DataFrame 'feature_importance_set_df' has been successfully loaded.


In [59]:
feature_importance_set_df

,Features,Importance,Importance_abs,feature_selector_idx
0,Zb_V_OBVOL_VSS_H of 1103959_69_1133592_QPP,0.000000,0.000000,0
1,AC_SENSE_CLP of 1103959_69_1133529_cp1_cp1p5_Y...,0.000000,0.000000,0
2,AC_OFFCALRES of 1103959_69_1133529_cp1,0.000000,0.000000,0
3,AC_SENSE_CN_FG_SA of 1103959_69_1133529_cp1,0.000000,0.000000,0
4,AC_SENSE_CN_FG_SA of 1103959_69_1133529_cp1_cp...,0.000000,0.000000,0
...,...,...,...,...
3790,AG_V_SELF_GAIN_7_4_1_N of 1103959_69_1133529_c...,0.039792,0.039792,3
3791,EEPROM_S3__,0.047992,0.047992,3
3792,AC_VOUTB0 of 1103959_69_1133529_cp1p5,0.057490,0.057490,3
3793,COL of 1103959_69_1133529_cp1,0.076079,0.076079,3


In [60]:
import pandas as pd
import plotly.express as px
import os

# feature_importance_set_df가 이미 로드되었다고 가정합니다.

# 1. 'feature_selector_idx'가 0인 데이터 필터링
filtered_df = feature_importance_set_df[feature_importance_set_df['feature_selector_idx'] == 0].copy()

# 2. 'Importance_abs' 기준으로 내림차순 정렬
sorted_df = filtered_df.sort_values(by='Importance_abs', ascending=False)

# 3. 상위 20개 특성 선택
top_20_df = sorted_df.head(20)

# 4. 가로 막대 차트 생성
fig = px.bar(
    top_20_df,
    x='Importance',
    y='Features',
    orientation='h',
    title='Feature Importance',
    labels={'Importance': 'Importance', 'Features': 'Feature'}
)

# 5. 레이아웃 업데이트 (가로 사이즈를 1200으로 설정)
# 이 설정이 주피터 노트북 출력과 파일 저장 모두에 동일하게 적용됩니다.
fig.update_layout(
    yaxis={'categoryorder': 'total ascending'},
    xaxis_title='Importance',
    yaxis_title='Feature',
    width=1500,  # Ensure this value matches your desired notebook display width
    height=500,
    margin=dict(l=50, r=50, t=50, b=50),
    title_x=0.5
)

# 이미지 파일로 저장
save_path = "data/result/select_feature/feature_importance_wide.png"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# fig.write_image(save_path)
# Save the image file, explicitly setting dimensions
fig.write_image(
    save_path, 
    width=1500, 
    height=500
)

# 차트 표시
fig.show()

print(f"가로 사이즈가 확장된 특성 중요도 차트가 {save_path}에 생성 및 저장되었습니다.")

가로 사이즈가 확장된 특성 중요도 차트가 data/result/select_feature/feature_importance_wide.png에 생성 및 저장되었습니다.


##### Model Parameter optimization with optuna

In [ ]:
# # feature select test : pipeline
# ## Apply the selected feature set feature_selection_info['final_features'] list to the modeling pipeline and return the modeling results

# def pl_fs_test(
#         train_data,
#         feature_selection_info,
#         train_parameters,
#         test_data
#         ):
    
#     trained_model, feature_importance, train_parameters_info = train_model_rf_optuna(train_data.copy(), feature_selection_info, train_parameters)
#     forecast_dataset, shap_values_random_forest = forecast(test_data, trained_model, feature_selection_info)
#     train_dataset_proba, best_threshold = find_best_threshold(trained_model, train_data.copy(), feature_selection_info)
#     roc_data, auc_score = roc_from_scratch(forecast_dataset, test_data, partitions=100)
#     train_dataset_metrics = create_metrics_on_train(train_dataset_proba, best_threshold)
#     metrics = create_metrics(forecast_dataset, test_data, auc_score, best_threshold)
#     results = create_results(forecast_dataset, test_data, best_threshold)
    
#     return \
#         trained_model, \
#         feature_importance, \
#         forecast_dataset, \
#         train_dataset_proba, \
#         best_threshold, \
#         roc_data, \
#         auc_score, \
#         train_dataset_metrics, \
#         metrics, \
#         results


In [ ]:
# # Optimal feature set performance check
# # feature select test - submit and summary result

# opt_pl_fs_test_result_ftpn_df_rf_optuna = pd.DataFrame()
# opt_pl_fs_test_result_features_values_dfs_rf_optuna = pd.DataFrame(
#     list(train_data.columns), 
#     columns=['feature_name']
# )

# feature_importance_set = pd.DataFrame()

# for fileter_name, feature_selection_info in feature_selection_results.items():
#     trained_model, \
#     feature_importance, \
#     forecast_dataset, \
#     train_dataset_proba, \
#     best_threshold, \
#     roc_data, \
#     auc_score, \
#     train_dataset_metrics, \
#     metrics, \
#     results \
#     = \
#     pl_fs_test(
#             train_data,
#             feature_selection_info,
#             train_parameters_list_default["rf_optuna"],
#             test_data
#             )

#     dict_ftpn = metrics["dict_ftpn"]
    
#     new_row = {
#         'fn': dict_ftpn.get('fn'),
#         'fp': dict_ftpn.get('fp'),
#         'tn': dict_ftpn.get('tn'),
#         'tp': dict_ftpn.get('tp'),
#         'feature_selector_idx' : feature_selection_info["feature_selector_idx"],
#         'feature_selector_name' : feature_selection_info["feature_selector_name"],
#         'initial_feature_count' : feature_selection_info["initial_feature_count"],
#         'final_feature_count' : feature_selection_info["final_feature_count"],
#         'final_feature_selector_f2score' : feature_selection_info["Params"]["f2_score"],
#         'final_feature_selector_best_k_percentile' : feature_selection_info["Params"]["best_k_percentile"]
#     }
    
#     opt_pl_fs_test_result_ftpn_df_rf_optuna = pd.concat([opt_pl_fs_test_result_ftpn_df_rf_optuna, pd.DataFrame([new_row])], ignore_index=True)
#     features_values = feature_importance[["Features", "Importance"]].rename(columns={"Importance": "Importance(model)"+"_"+feature_selection_info["feature_selector_name"] })

#     opt_pl_fs_test_result_features_values_dfs_rf_optuna = pd.merge(
#         opt_pl_fs_test_result_features_values_dfs_rf_optuna,
#         features_values,
#         how='left',
#         left_on='feature_name',
#         right_on='Features'
#     ).drop('Features', axis=1) # The .drop() method is used to drop the column
